In [2]:
import pandas as pd
import requests
import os
import zipfile

# Define o diretório atual e cria uma pasta para os dados
diretorio_atual = os.getcwd()
os.chdir(diretorio_atual)
diretorio_dados = "dados_cvm" 
os.chdir(os.path.join(diretorio_atual, diretorio_dados))

In [2]:
anos = range(2010, 2026)
url_base = "http://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/"

for ano in anos:
    arquivo = f"dfp_cia_aberta_{ano}.zip"
    download = requests.get(url_base + arquivo)
    
    # Salva o conteúdo no computador
    with open(arquivo, "wb") as f:
        f.write(download.content)

KeyboardInterrupt: 

In [3]:
lista_demonstracoes = []

diretorio_arquivos = os.getcwd()

for arquivo in os.listdir(diretorio_arquivos):
    # Correção: Strings nativas do Python não usam .str
    if arquivo.endswith(".zip"):
        with zipfile.ZipFile(arquivo) as arquivo_zip:
            for planilha in arquivo_zip.namelist():
                # Lê o CSV com separador específico da CVM e encoding para caracteres em português
                with arquivo_zip.open(planilha) as f:
                    demonstracao = pd.read_csv(f, sep=';', encoding='ISO-8859-1')
                    lista_demonstracoes.append(demonstracao)

# Consolida tudo em um único DataFrame apenas se houver dados
if lista_demonstracoes:
    base_dados = pd.concat(lista_demonstracoes, ignore_index=True)
    print("Dados consolidados com sucesso!")
else:
    print("Nenhum arquivo .zip processado.")

Dados consolidados com sucesso!


In [3]:
base_dados = pd.read_csv('CVM_bruto.csv')

/tmp/ipykernel_15374/1212410393.py:1: DtypeWarning: Columns (0: CATEG_DOC, 1: DT_RECEB, 2: LINK_DOC, 3: DT_INI_EXERC, 4: COLUNA_DF, 5: TP_RELAT_AUD, 6: TP_PARECER_DECL, 7: TXT_PARECER_DECL) have mixed types. Specify dtype option on import or set low_memory=False.
  base_dados = pd.read_csv('CVM_bruto.csv')


In [7]:
base_dados.head()

print(base_dados.columns.values)

<StringArray>
[                'CNPJ_CIA',                 'DT_REFER',
                   'VERSAO',                'DENOM_CIA',
                   'CD_CVM',                'CATEG_DOC',
                   'ID_DOC',                 'DT_RECEB',
                 'LINK_DOC',                'GRUPO_DFP',
                    'MOEDA',             'ESCALA_MOEDA',
              'ORDEM_EXERC',             'DT_FIM_EXERC',
                 'CD_CONTA',                 'DS_CONTA',
                 'VL_CONTA',            'ST_CONTA_FIXA',
             'DT_INI_EXERC',                'COLUNA_DF',
             'TP_RELAT_AUD',          'TP_PARECER_DECL',
    'NUM_ITEM_PARECER_DECL',         'TXT_PARECER_DECL',
 'QT_ACAO_ORDIN_CAP_INTEGR',  'QT_ACAO_PREF_CAP_INTEGR',
 'QT_ACAO_TOTAL_CAP_INTEGR',    'QT_ACAO_ORDIN_TESOURO',
     'QT_ACAO_PREF_TESOURO',    'QT_ACAO_TOTAL_TESOURO']
Length: 30, dtype: str


In [4]:
# Mais eficiente: faz o split uma vez e indexa cada parte
split = base_dados['GRUPO_DFP'].str.split('-', n=1)
base_dados['cond_ind'] = split.str[0].str.strip()
base_dados['tipo_dem'] = split.str[1].str.strip()
del split  # libera memória imediatamente

base_dados = base_dados[base_dados['ORDEM_EXERC'] != "PENÚLTIMO"]

In [9]:
lista_dem = base_dados['tipo_dem'].unique() # talvez seja tipo_dem

lista_empresas = base_dados['DENOM_CIA'].unique()

lista_empresas

<StringArray>
[                                           'BCO BRASIL S.A.',
                                 'BRB BANCO DE BRASILIA S.A.',
                       'CENTRAIS ELET BRAS S.A. - ELETROBRAS',
                     'COMPANHIA ENERGÉTICA DE BRASÍLIA - CEB',
                                'SHOPPING CENTER TACARUNA SA',
                    'NEUMARKT TRADE AND FINANCIAL CENTER S/A',
                                                  'CIMS S.A.',
                            'TELEC BRASILEIRAS S.A. TELEBRAS',
 'KOSMOS COMÉRCIO DE VESTUÁRIO S/A - EM RECUPERAÇÃO JUDICIAL',
                  'ATOM EMPREENDIMENTOS E PARTICIPAÇÕES S.A.',
 ...
                             'MERCANTIL DO BRASIL LEASING SA',
                                 'BANCO INDUSTRIAL DO BRASIL',
                                          'CIMENTO TUPI S.A.',
                     'JPSP INVESTIMENTOS E PARTICIPAÇÕES S/A',
                                 'FIBAM COMPANHIA INDUSTRIAL',
                    'SAFRA LEASING S

In [15]:
df = pd.DataFrame(lista_empresas, columns=['Lista empresas'])
df.to_csv('lista_empres.csv', index=False)

In [5]:
base_dados.to_csv('CVM.csv', index = False)

In [11]:
weg_receita = base_dados[
    (base_dados["DENOM_CIA"] == 'WEG S.A.') &
    (base_dados["tipo_dem"] == "Demonstração do Resultado") &   # ← parênteses próprios
    (base_dados["DS_CONTA"] == 'Receita de Venda de Bens e/ou Serviços') &
    (base_dados['cond_ind'] == 'DF Consolidado')
]

print(weg_receita)

                  CNPJ_CIA    DT_REFER  VERSAO DENOM_CIA  CD_CVM CATEG_DOC  \
825795  84.429.695/0001-11  2018-12-31       2  WEG S.A.  5410.0       NaN   

        ID_DOC DT_RECEB LINK_DOC                                   GRUPO_DFP  \
825795     NaN      NaN      NaN  DF Consolidado - Demonstração do Resultado   

        ... NUM_ITEM_PARECER_DECL TXT_PARECER_DECL QT_ACAO_ORDIN_CAP_INTEGR  \
825795  ...                   NaN              NaN                      NaN   

       QT_ACAO_PREF_CAP_INTEGR QT_ACAO_TOTAL_CAP_INTEGR QT_ACAO_ORDIN_TESOURO  \
825795                     NaN                      NaN                   NaN   

        QT_ACAO_PREF_TESOURO QT_ACAO_TOTAL_TESOURO        cond_ind  \
825795                   NaN                   NaN  DF Consolidado   

                         tipo_dem  
825795  Demonstração do Resultado  

[1 rows x 32 columns]
